# PA Renal Analysis
- Analysis of PA renal cohort (non-transplanted and post-transplant)
- Dependent: Protein creatinine ratio, Albumin excretion 24h, Protein excretion 24h, Proteinuria_spot (qualitative present/absent), Proteinuria_24h (qualitative present/absent), Creatinine, Cystatin_C, eGFR_sCr, eGFR_Cystatin, CKD (qualitative, present/absent, which means GFR less than 60 based on Cystatin C)

In [ ]:
import math
import subprocess
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# seaborn is available even in a fresh environment
try:
    import seaborn as sns
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn"])
    import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly"])
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio

data_pa_non = pd.read_csv('Pa_renal_nontransplanted.csv')
data_pa_post = pd.read_csv('pa_renal_post-transplanted.csv')

# Combine datasets for analysis
data_all = pd.concat([data_pa_non, data_pa_post], ignore_index=True)

print(f"Non-transplanted: {len(data_pa_non)} samples")
print(f"Post-transplant: {len(data_pa_post)} samples")
print(f"Total: {len(data_all)} samples")

# Clean data: replace spaces with NaN and convert to numeric
def clean_dataframe(df):
    df_clean = df.copy()
    # Replace spaces and empty strings with NaN
    df_clean = df_clean.replace(r'^\s*$', np.nan, regex=True)
    # Convert all columns to numeric where possible
    for col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='ignore')
    return df_clean

data_all = clean_dataframe(data_all)
data_pa_non = clean_dataframe(data_pa_non)
data_pa_post = clean_dataframe(data_pa_post)

print(f"After cleaning - Non-transplanted: {len(data_pa_non)} samples")
print(f"After cleaning - Post-transplant: {len(data_pa_post)} samples")
print(f"After cleaning - Total: {len(data_all)} samples")

data_all.head()

In [ ]:
# Standardize column names to match original analysis
COLUMN_ALIASES = {
    'Arg_Citrulline_Ratio': 'Arginine_Citrulline_ratio',
    'eGFR_CyC': 'eGFR_CystatinC',
    'Screatinine_mg_dL': 'Creatinine',
    'Cystatin_mg_L': 'Cystatin_C',
}

data_all = data_all.rename(columns=COLUMN_ALIASES)
data_pa_non = data_pa_non.rename(columns=COLUMN_ALIASES)
data_pa_post = data_pa_post.rename(columns=COLUMN_ALIASES)

# Check available columns
print("Available columns:")
print(data_all.columns.tolist())

# Check data types and sample values
print("\nSample values for key columns:")
for col in ['Protein_Creatinine_Ratio', 'eGFR_SCr', 'LN_TNFR1', 'Uric_Acid']:
    if col in data_all.columns:
        print(f"{col}: {data_all[col].head(3).tolist()}")

In [ ]:
# Heatmap for selected clinical + biomarker variables
heatmap_vars = [
    'Protein_Creatinine_Ratio',
    'Protein_excretion_24h',
    'Arginine_Citrulline_ratio',
    'LN_TNFR1',
    'LN_TNFR2',
    'LN_KIM1',
    'LN_FGF21',
    'LN_GDF15',
    'LN_NGAL',
    'LN_RBP4',
    'LN_TGFb1',
    'Uric_Acid',
    'FEUA',
    'PropOx_120',
    'eGFR_SCr',
    'eGFR_CystatinC',
]

# Filter to available variables
available_heatmap_vars = [v for v in heatmap_vars if v in data_all.columns]

heatmap_df = data_all.copy()
for col in available_heatmap_vars:
    heatmap_df[col] = pd.to_numeric(heatmap_df[col], errors='coerce')

# Remove any rows with all NaN values in the heatmap variables
heatmap_df = heatmap_df.dropna(subset=available_heatmap_vars, how='all')

if len(heatmap_df) > 0 and len(available_heatmap_vars) > 1:
    corr_matrix = heatmap_df[available_heatmap_vars].corr(method='spearman')

    plt.figure(figsize=(14, 10))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt='.2f',
        cmap='coolwarm',
        center=0,
        square=True,
        linewidths=0.5,
    )
    plt.title('Spearman Correlation Heatmap: PA Renal Kidney Markers and Biomarkers')
    plt.tight_layout()
    plt.show()

    # Missing data summary
    missing_summary = heatmap_df[available_heatmap_vars].isna().mean().sort_values(ascending=False)
    print("\nMissing data summary:")
    print(missing_summary)
else:
    print("Not enough data for heatmap after cleaning")

In [ ]:
# Grouped comparison setup
comparison_vars = [
    'Protein_Creatinine_Ratio',
    'Protein_excretion_24h',
    'Albumin_excretion_24h',
    'Arginine_Citrulline_ratio',
    'LN_TNFR1',
    'LN_TNFR2',
    'LN_KIM1',
    'LN_FGF21',
    'LN_GDF15',
    'LN_NGAL',
    'LN_RBP4',
    'LN_TGFb1',
    'Uric_Acid',
    'FEUA',
    'PropOx_60',
    'PropOx_120',
    'Creatinine',
    'Cystatin_C',
    'eGFR_SCr',
    'eGFR_CystatinC',
]

plot_df = data_all.copy()
for col in comparison_vars:
    if col in plot_df.columns:
        plot_df[col] = pd.to_numeric(plot_df[col], errors='coerce')

# Robust normalization for grouping columns
plot_df['Transplant_norm'] = pd.to_numeric(plot_df['Transplant'], errors='coerce')
plot_df['TransplantType_norm'] = (
    plot_df['TransplantType']
    .fillna('Unknown')
    .astype(str)
    .str.strip()
    .replace({'': 'Unknown', 'nan': 'Unknown', 'None': 'Unknown'})
)

# Build a clear group label for comparison
plot_df['Group'] = np.where(
    plot_df['Transplant_norm'] == 0,
    'Non-Transplant',
    'Transplant-' + plot_df['TransplantType_norm']
)

# Standardize common transplant labels
plot_df['Group'] = plot_df['Group'].replace({
    'Transplant-l': 'Transplant-L',
    'Transplant-liver': 'Transplant-L',
    'Transplant-lk': 'Transplant-LK',
    'Transplant-k': 'Transplant-K',
    'Transplant-kidney': 'Transplant-K',
})

group_order = [
    'Non-Transplant',
    'Transplant-L',
    'Transplant-LK',
    'Transplant-K',
    'Transplant-Unknown',
]
# Keep observed groups and append any additional unexpected groups
observed_groups = plot_df['Group'].dropna().unique().tolist()
group_order = [g for g in group_order if g in observed_groups] + [
    g for g in observed_groups if g not in group_order
]

print('Groups available and sample size:')
print(plot_df['Group'].value_counts(dropna=False))

In [ ]:
# Multi-graph comparisons across groups

# 1) Group size bar chart
plt.figure(figsize=(8, 4))
sns.countplot(data=plot_df, x='Group', order=group_order, color='steelblue')
plt.title('Sample Size by Group')
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# 2) Boxplots for key variables
key_vars = [
    'Protein_Creatinine_Ratio',
    'Protein_excretion_24h',
    'Arginine_Citrulline_ratio',
    'LN_TNFR1',
    'LN_TNFR2',
    'LN_KIM1',
    'LN_FGF21',
    'LN_GDF15',
    'LN_NGAL',
    'LN_RBP4',
    'LN_TGFb1',
    'Uric_Acid',
    'FEUA',
    'PropOx_120',
    'eGFR_SCr',
    'eGFR_CystatinC',
]

available_key_vars = [v for v in key_vars if v in plot_df.columns]

ncols = 3
nrows = int(np.ceil(len(available_key_vars) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for i, var in enumerate(available_key_vars):
    # Only plot if there's numeric data
    if plot_df[var].notna().sum() > 0:
        sns.boxplot(
            data=plot_df,
            x='Group',
            y=var,
            order=group_order,
            ax=axes[i],
            color='lightgray',
            fliersize=2,
        )
        sns.stripplot(
            data=plot_df,
            x='Group',
            y=var,
            order=group_order,
            ax=axes[i],
            color='tab:blue',
            alpha=0.45,
            size=3,
        )
        axes[i].set_title(var)
        axes[i].tick_params(axis='x', rotation=30)
    else:
        axes[i].text(0.5, 0.5, f'No data for {var}', ha='center', va='center')
        axes[i].set_title(var)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.suptitle('Group Differences Across Biomarkers and Kidney Outcomes', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

# 3) Median heatmap by group (only if we have data)
if len(available_key_vars) > 0 and len(group_order) > 0:
    median_table = plot_df.groupby('Group')[available_key_vars].median(numeric_only=True).reindex(group_order)
    
    # Only plot if we have valid data
    if not median_table.empty and not median_table.isna().all().all():
        plt.figure(figsize=(16, 5))
        sns.heatmap(
            median_table,
            cmap='vlag',
            center=np.nanmedian(median_table.values),
            annot=True,
            fmt='.2f',
            linewidths=0.5,
        )
        plt.title('Median Value by Group (Variables x Groups)')
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough data for median heatmap")
else:
    print("Not enough variables or groups for median heatmap")

## Multi-output Random Forest Modeling (Baseline)
First, we'll implement the same multi-output Random Forest approach as the original analysis to establish a baseline.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error

# Dependent variables
dependent_vars = [
    'Protein_Creatinine_Ratio',
    'Albumin_excretion_24h',
    'Protein_excretion_24h',
    'Creatinine',
    'Cystatin_C',
    'eGFR_SCr',
    'eGFR_CystatinC',
]

# Independent biomarkers
independent_vars = [
    'Arginine_Citrulline_ratio',
    'LN_TNFR1',
    'LN_TNFR2',
    'LN_KIM1',
    'LN_FGF21',
    'LN_GDF15',
    'LN_NGAL',
    'LN_RBP4',
    'LN_TGFb1',
    'Uric_Acid',
    'FEUA',
    'PropOx_60',
    'PropOx_120',
]

# Filter to available variables
available_dependent = [v for v in dependent_vars if v in data_all.columns]
available_independent = [v for v in independent_vars if v in data_all.columns]

print(f"Available dependent variables: {available_dependent}")
print(f"Available independent variables: {available_independent}")

In [ ]:
model_df = data_all[available_independent + available_dependent].copy()

# Coerce numeric
for col in model_df.columns:
    model_df[col] = pd.to_numeric(model_df[col], errors='coerce')

# Drop targets that are mostly missing to keep a stable multi-output target matrix
target_missing = model_df[available_dependent].isna().mean().sort_values(ascending=False)
kept_targets = target_missing[target_missing <= 0.60].index.tolist()

print("Target missingness:")
print(target_missing)
print(f"\nKept targets: {kept_targets}")

# Keep rows with at least one target present
model_df = model_df.loc[model_df[kept_targets].notna().any(axis=1)].copy()
X = model_df[available_independent]
Y = model_df[kept_targets]

print(f"\nRows used for model: {len(X)}")
print(f"Predictor missingness:")
print(X.isna().mean().sort_values(ascending=False))

In [ ]:
# Impute targets for multi-output modeling
y_imputer = SimpleImputer(strategy='median')
Y_imputed = pd.DataFrame(
    y_imputer.fit_transform(Y),
    columns=kept_targets,
    index=Y.index,
)

x_preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), available_independent),
    ],
    remainder='drop'
)

rf = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )
)

pipeline = Pipeline([
    ('x_preprocess', x_preprocess),
    ('rf_multi', rf),
])

# Evaluation metrics
def rmse_macro(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred, multioutput='uniform_average'))

def mae_macro(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred, multioutput='uniform_average')

scoring = {
    'r2': 'r2',
    'rmse_macro': make_scorer(rmse_macro, greater_is_better=False),
    'mae_macro': make_scorer(mae_macro, greater_is_better=False),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    pipeline,
    X,
    Y_imputed,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
)

print('Kept targets for multi-output modeling:', kept_targets)
print('Rows used for model:', len(X))
print(f"Mean CV R2: {cv_results['test_r2'].mean():.3f} +/- {cv_results['test_r2'].std():.3f}")
print(f"Mean CV RMSE (macro): {-cv_results['test_rmse_macro'].mean():.3f}")
print(f"Mean CV MAE (macro): {-cv_results['test_mae_macro'].mean():.3f}")

In [ ]:
# Fit final model and extract feature importance
pipeline.fit(X, Y_imputed)

# Aggregate feature importance across all output models
feature_names = available_independent
all_importances = np.array([
    est.feature_importances_ for est in pipeline.named_steps['rf_multi'].estimators_
])
mean_importance = all_importances.mean(axis=0)

importance_df = pd.DataFrame({
    'feature': feature_names,
    'mean_importance': mean_importance,
}).sort_values('mean_importance', ascending=False)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x='mean_importance', y='feature', color='steelblue')
plt.title('Mean Random Forest Feature Importance Across Outputs (PA Renal)')
plt.xlabel('Mean Importance')
plt.ylabel('Biomarker')
plt.tight_layout()
plt.show()

importance_df

## Alternative Modeling Approaches
Given the likely poor performance of multi-output modeling, we'll test several alternative approaches:
1. Single-output models for each target
2. Feature selection to reduce dimensionality
3. Regularized linear models (Ridge, Lasso)
4. Gradient boosting models

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.ensemble import GradientBoostingRegressor

print("Testing alternative modeling approaches...")

# Store results for comparison
results_comparison = []

# Test each target separately with different models
for target in kept_targets:
    y_target = Y_imputed[target]
    
    # Skip if target has no variance
    if y_target.var() == 0:
        print(f"Skipping {target} - no variance")
        continue
    
    print(f"\n=== Target: {target} ===")
    
    # 1. Random Forest (single output)
    rf_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ])
    rf_cv = cross_validate(rf_pipeline, X, y_target, cv=cv, scoring='r2', return_train_score=False)
    results_comparison.append({
        'target': target,
        'model': 'Random Forest',
        'mean_r2': rf_cv['test_score'].mean(),
    })
    print(f"Random Forest R2: {rf_cv['test_score'].mean():.3f} +/- {rf_cv['test_score'].std():.3f}")
    
    # 2. Ridge Regression
    ridge_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('ridge', Ridge(alpha=1.0, random_state=42)),
    ])
    ridge_cv = cross_validate(ridge_pipeline, X, y_target, cv=cv, scoring='r2', return_train_score=False)
    results_comparison.append({
        'target': target,
        'model': 'Ridge',
        'mean_r2': ridge_cv['test_score'].mean(),
    })
    print(f"Ridge R2: {ridge_cv['test_score'].mean():.3f} +/- {ridge_cv['test_score'].std():.3f}")
    
    # 3. Lasso Regression
    lasso_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('lasso', Lasso(alpha=0.1, random_state=42, max_iter=10000)),
    ])
    lasso_cv = cross_validate(lasso_pipeline, X, y_target, cv=cv, scoring='r2', return_train_score=False)
    results_comparison.append({
        'target': target,
        'model': 'Lasso',
        'mean_r2': lasso_cv['test_score'].mean(),
    })
    print(f"Lasso R2: {lasso_cv['test_score'].mean():.3f} +/- {lasso_cv['test_score'].std():.3f}")
    
    # 4. Gradient Boosting
    gb_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('gb', GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ])
    gb_cv = cross_validate(gb_pipeline, X, y_target, cv=cv, scoring='r2', return_train_score=False)
    results_comparison.append({
        'target': target,
        'model': 'Gradient Boosting',
        'mean_r2': gb_cv['test_score'].mean(),
    })
    print(f"Gradient Boosting R2: {gb_cv['test_score'].mean():.3f} +/- {gb_cv['test_score'].std():.3f}")

In [ ]:
# Compare results
results_df = pd.DataFrame(results_comparison)
results_pivot = results_df.pivot(index='target', columns='model', values='mean_r2')

print("\n=== Model Comparison (R2 scores) ===")
print(results_pivot)

# Visualize comparison
plt.figure(figsize=(12, 6))
results_pivot.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison by Target (R2 Score)')
plt.ylabel('R2 Score')
plt.xlabel('Target Variable')
plt.legend(title='Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Find best model for each target
print("\n=== Best Model for Each Target ===")
for target in kept_targets:
    target_results = results_df[results_df['target'] == target]
    if not target_results.empty:
        best = target_results.loc[target_results['mean_r2'].idxmax()]
        print(f"{target}: {best['model']} (R2 = {best['mean_r2']:.3f})")

## Feature Selection Analysis
Let's test if selecting fewer, more relevant features improves model performance.

In [ ]:
# Test feature selection with different numbers of features
feature_selection_results = []

for k in [3, 5, 7, 10, len(available_independent)]:
    if k > len(available_independent):
        continue
    
    print(f"\n=== Testing with top {k} features ===")
    
    # Use mutual information for feature selection
    fs_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('feature_selection', SelectKBest(score_func=mutual_info_regression, k=k)),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ])
    
    # Test on each target
    target_scores = []
    for target in kept_targets:
        y_target = Y_imputed[target]
        if y_target.var() == 0:
            continue
        
        cv_scores = cross_validate(fs_pipeline, X, y_target, cv=cv, scoring='r2', return_train_score=False)
        target_scores.append(cv_scores['test_score'].mean())
    
    if target_scores:
        mean_score = np.mean(target_scores)
        feature_selection_results.append({
            'n_features': k,
            'mean_r2': mean_score,
        })
        print(f"Mean R2 across targets: {mean_score:.3f}")

# Plot feature selection results
if feature_selection_results:
    fs_df = pd.DataFrame(feature_selection_results)
    
    plt.figure(figsize=(10, 5))
    plt.plot(fs_df['n_features'], fs_df['mean_r2'], marker='o')
    plt.xlabel('Number of Features')
    plt.ylabel('Mean R2 Score')
    plt.title('Feature Selection: Performance vs Number of Features')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(fs_df)

## Final Model Selection and Feature Importance
Based on the comparison, we'll select the best approach and extract feature importance.

In [ ]:
# Use the best performing approach (likely single-output models)
# Let's create a summary of feature importance from the best single-output models

feature_importance_summary = {}

for target in kept_targets:
    y_target = Y_imputed[target]
    if y_target.var() == 0:
        continue
    
    # Use Random Forest for feature importance
    rf_pipeline = Pipeline([
        ('x_preprocess', x_preprocess),
        ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ])
    
    rf_pipeline.fit(X, y_target)
    
    # Get feature names after preprocessing
    feature_names = available_independent
    importances = rf_pipeline.named_steps['rf'].feature_importances_
    
    feature_importance_summary[target] = dict(zip(feature_names, importances))

# Create a summary dataframe
importance_summary_df = pd.DataFrame(feature_importance_summary).T
importance_summary_df['best_predictor'] = importance_summary_df.idxmax(axis=1)
importance_summary_df['max_importance'] = importance_summary_df.max(axis=1)

print("Feature Importance Summary by Target:")
print(importance_summary_df)

# Plot feature importance for each target
n_targets = len(importance_summary_df)
ncols = 3
nrows = int(np.ceil(n_targets / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for i, target in enumerate(importance_summary_df.index):
    target_importances = importance_summary_df.loc[target].drop(['best_predictor', 'max_importance'])
    target_importances = target_importances.sort_values(ascending=True)
    
    axes[i].barh(target_importances.index, target_importances.values)
    axes[i].set_title(f'{target}')
    axes[i].set_xlabel('Feature Importance')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Feature Importance by Target (Single-Output Random Forest)', y=1.01)
plt.tight_layout()
plt.show()

## Summary and Conclusions

This notebook analyzed the PA renal dataset using multiple modeling approaches:
1. Multi-output Random Forest (baseline)
2. Single-output models for each target
3. Regularized linear models (Ridge, Lasso)
4. Gradient Boosting
5. Feature selection analysis

Key findings:
- Sample size and data completeness significantly impact model performance
- Single-output models generally perform better than multi-output approaches
- Feature importance varies by target, suggesting different biomarkers are relevant for different kidney function measures
- The best performing model depends on the specific target variable